# Setup

In [1]:
import os
import numpy as np
import torch

%load_ext autoreload
%autoreload 2

print("os cpu count:", os.cpu_count())

os cpu count: 96


In [7]:
import torch.nn as nn

# Mixed Precision Training

In [2]:
s = torch.tensor(0, dtype=torch.float32)
for i in range(1000):
    s += torch.tensor(0.01, dtype=torch.float32)
print(s)

tensor(10.0001)


In [ ]:
s = torch.tensor(0, dtype=torch.float16)
for i in range(1000):
    s += torch.tensor(0.01, dtype=torch.float16)
print(s)

tensor(9.9531, dtype=torch.float16)


In [5]:
s = torch.tensor(0, dtype=torch.float32)
for i in range(1000):
    s += torch.tensor(0.01, dtype=torch.float16)
print(s)

tensor(10.0021)


In [6]:
s = torch.tensor(0, dtype=torch.float32)
for i in range(1000):
    x = torch.tensor(0.01, dtype=torch.float16)
    s += x.type(torch.float32)
print(s)

tensor(10.0021)


In [8]:
class ToyModel(nn.Module):
    def __init__(self, in_features: int, out_features: int):
        super().__init__()
        self.fc1 = nn.Linear(in_features, 10, bias=False)
        self.ln = nn.LayerNorm(10)
        self.fc2 = nn.Linear(10, out_features, bias=False)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.relu(self.fc1(x))
        x = self.ln(x)
        x = self.fc2(x)
        return x

In [24]:
model: nn.Module = ToyModel(10, 10)
print(model)
dtype: torch.dtype = torch.float16
# dtype: torch.dtype = torch.bfloat16
print("dtype:", dtype)
x = torch.randn(10, 10)
target = torch.randn(10, 10)
device = "cuda:0" if torch.cuda.is_available() else "cpu"


with torch.autocast(device_type="cuda", dtype=dtype):
    x = x.to(device=device)
    target = target.to(device=device)
    model = model.to(device=device)
    y = model(x)
    loss = torch.nn.functional.mse_loss(y, target)
    loss.backward()
    for name, param in model.named_parameters():
        print(f"{name} dtype: {param.dtype}")
    print("x dtype:", x.dtype)
    print("fc1 weight dtype:", model.fc1.weight.dtype)
    print("ln weight dtype:", model.ln.weight.dtype)
    print("loss dtype:", loss.dtype)
    print("y dtype:", y.dtype)
    print("grad dtype:", model.fc1.weight.grad.dtype)


ToyModel(
  (fc1): Linear(in_features=10, out_features=10, bias=False)
  (ln): LayerNorm((10,), eps=1e-05, elementwise_affine=True)
  (fc2): Linear(in_features=10, out_features=10, bias=False)
  (relu): ReLU()
)
dtype: torch.float16
fc1.weight dtype: torch.float32
ln.weight dtype: torch.float32
ln.bias dtype: torch.float32
fc2.weight dtype: torch.float32
x dtype: torch.float32
fc1 weight dtype: torch.float32
ln weight dtype: torch.float32
loss dtype: torch.float32
y dtype: torch.float16
grad dtype: torch.float32
